In [47]:
import numpy as np
import pandas as pd
import datetime
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sns

In [48]:
df = pd.read_excel('urbox_bi_analyst_test.xlsx', sheet_name = 'Data for Part 1')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5652 entries, 0 to 5651
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   transaction_id       5652 non-null   int64 
 1   user_id              5652 non-null   int64 
 2   brand_id             5652 non-null   int64 
 3   voucher_redeemed_at  5652 non-null   object
dtypes: int64(3), object(1)
memory usage: 176.8+ KB


In [76]:
df['voucher_redeemed_at'] = pd.to_datetime(df['voucher_redeemed_at'])

# 1. Tính toán UrBox Age
user_dates = df.groupby('user_id')['voucher_redeemed_at'].agg(['min', 'max'])

# Phép trừ này bây giờ sẽ hoạt động vì dữ liệu đã là Datetime
user_dates['urbox_age'] = (user_dates['max'] - user_dates['min']).dt.days

# 2. Phân nhóm 
def segment(age):
    if age == 0: return 'Within_a_day'
    if age <= 30: return '1 month'
    if age <= 90: return '1-3 months'
    return '>3 months'

user_dates['age_group'] = user_dates['urbox_age'].apply(segment)

# 3. Phân tích Cluster
merged = df.merge(user_dates[['age_group']], left_on='user_id', right_index=True)
cluster_analysis = merged.groupby(['age_group', 'brand_id'], as_index = False)['user_id'].agg(['count'])

In [77]:
Within_a_day_group = cluster_analysis[cluster_analysis['age_group'] == 'Within_a_day'].sort_values(by = 'count', ascending = False)
Within_a_day_group['percentage'] = (Within_a_day_group['count']/ Within_a_day_group['count'].sum() * 100).round(2)
Within_a_day_group.head(5)

,age_group,brand_id,count,percentage
194,Within_a_day,1511,219,32.93
170,Within_a_day,395,78,11.73
159,Within_a_day,82,69,10.38
187,Within_a_day,910,37,5.56
171,Within_a_day,396,30,4.51


Từ bảng trên cho thấy với nhóm sử dụng hết voucher trong ngày thì brand id 1511(32.9%), 395 (11.7%) và 82 (10.3%) chiếm tỷ trọng lớn, trong đó brand id 1511 chiếm chủ yếu => Điều này cho thấy khách có xu hướng tìm tới 3 brands này đầu tiên

In [80]:
one_month_group = cluster_analysis[cluster_analysis['age_group'] == '1 month'].sort_values(by = 'count', ascending = False)
one_month_group['percentage'] = (one_month_group['count']/ one_month_group['count'].sum() * 100).round(2)
one_month_group.head(5)

,age_group,brand_id,count,percentage
6,1 month,82,76,15.42
40,1 month,1511,72,14.60
15,1 month,395,56,11.36
2,1 month,27,33,6.69
30,1 month,882,32,6.49


Từ bảng trên ta thấy, với nhóm sử dụng hết voucher trong 1 tháng thì 3 brands 1511, 82 và 395 vẫn là chiếm tỷ trọng lớn, tuy nhiên có thể thấy số lần mua ở brand 1511 giảm đi khá nhiều so với nhóm sử dụng trong ngày, trong khi đó brand 82 được sử dụng nhiều hơn trong khoảng thời gian này. 

In [81]:
three_month_group = cluster_analysis[cluster_analysis['age_group'] == '1-3 months'].sort_values(by = 'count', ascending = False)
three_month_group['percentage'] = (three_month_group['count']/ three_month_group['count'].sum() * 100).round(2)
three_month_group.head(5)

,age_group,brand_id,count,percentage
87,1-3 months,1511,268,26.04
50,1-3 months,82,125,12.15
65,1-3 months,552,86,8.36
59,1-3 months,395,75,7.29
76,1-3 months,882,70,6.80


Nhóm từ 1-3 tháng xuất hiện brand mới trong top 3 là 552, brand 1511 được quay trở lại sử dụng nhiều trong khoảng thời gian này, chiếm tới 26%, trong khi đó nhóm brand 82 vẫn được duy trì sử dụng cao dù là ở giai đoạn nào.

In [82]:
over_three_month_group = cluster_analysis[cluster_analysis['age_group'] == '>3 months'].sort_values(by = 'count', ascending = False)
over_three_month_group['percentage'] = (over_three_month_group['count']/ over_three_month_group['count'].sum() * 100).round(2)
over_three_month_group.head(5)

,age_group,brand_id,count,percentage
150,>3 months,1511,702,20.26
123,>3 months,552,621,17.92
99,>3 months,82,544,15.70
90,>3 months,7,147,4.24
132,>3 months,673,144,4.16


Từ bảng trên ta có thể thấy nhóm này chiếm số lượng sử dụng voucher nhiều nhất, brand 395 biến mất trong top 3 , chủ yếu vẫn là nhóm brand 1511, 552 và 82. 

Kết luận chung: ta có thể thấy là 
1/ Các brand được sủ dụng nhiều nhất là 1511, 82, 552, 395
2/ brand 395 chủ yếu được quy đổi hết trong giai đoạn 3 tháng kể từ ngày bắt đầu sử dụng voucher
3/ brand 1511 chiếm đa số trong tổng số giao dịch quy đổi voucher, sụt giảm nhẹ ở giai đoạn 1 tháng đầu tiên (có thể liên quan tới vòng đới sử dụng sản phẩm thuộc ngành hàng này).
4/ Nhớm 82 được quy đổi xuyên suốt và có nhu cầu thường trực. 
5/ Nhóm 552 được sử dụng nhiều sau 30 ngày đầu tiên